# 02g · Repo-vs-paper-text variants (sensitivity of each reported discrepancy)
Each variant changes ONE thing relative to the repo default and runs only the methods it affects (Table II protocol, 5 recovery rounds).
Results are stored as experiment `variant_<name>`; compare with the Table II default run (02a) of the same dataset.
See `docs/FIXES_AND_DEVIATIONS.md` for what each switch is about (the 14-point review).
Variants that do not touch training share the cached pre-unlearning checkpoint of the default run.

In [ ]:
import os, sys, pathlib, importlib.util, subprocess
_cwd = pathlib.Path.cwd().resolve()
ROOT = next((p for p in [_cwd, *_cwd.parents, pathlib.Path("/content/FuGuard_Reproduction")]
             if (p / "fuguard_repro").exists()), None)
if ROOT is None:
    raise RuntimeError("Project not found. On Colab: run COLAB_START_HERE.ipynb first (it unzips to /content/FuGuard_Reproduction).")
os.environ.setdefault("FUGUARD_ROOT", str(ROOT))
sys.path.insert(0, str(ROOT))
for _mod in ("geomloss", "diffusers"):          # a Colab runtime restart wipes pip installs -> re-install if missing
    if importlib.util.find_spec(_mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-colab.txt")], check=True)
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
        break
print("project root:", ROOT)

In [ ]:
from fuguard_repro.config import image_config
from fuguard_repro.data_image import prepare_image_data
from fuguard_repro.runner import run_grid, standard_jobs
from fuguard_repro.aggregate import table2_numeric, table2_df

DS = "CIFAR10"            # change freely; the EuroSAT trigger variant is always EuroSAT
SEEDS = (0, 1)
ALL = ["retrain", "fedsga", "fu", "not", "fastfedul", "fuguard"]
VARIANTS = {   # name: (dataset, config overrides, methods affected)
    "eurosat_white_trigger": ("EuroSAT", {"trigger_mode": "white"}, ALL),
    "local_sgd":             (DS, {"optimizer": "sgd"}, ALL),
    "unlearn_at_round_51":   (DS, {"unlearn_round": 51}, ALL),
    "inject_50pct":          (DS, {"inject_ratio": 0.5}, ALL),
    "baseline_bs_32":        (DS, {"baseline_bs": 32}, ["fedsga", "fu"]),
    "fuguard_bs_256":        (DS, {"unlearn_bs": 256}, ["fuguard"]),
    "no_proxy_clamp":        (DS, {"proxy_clamp": False}, ["fuguard"]),
    "not_first_layer_only":  (DS, {"not_layers": ["features.0"]}, ["not"]),
}
SELECTED = list(VARIANTS)      # e.g. ["eurosat_white_trigger"] to run just one

In [ ]:
for name in SELECTED:
    ds, ov, methods = VARIANTS[name]
    print("\n#####", name, ov)
    for seed in SEEDS:
        prepare_image_data(image_config(ds, **ov), seed)          # no-op unless the data differs (trigger / inject ratio)
    run_grid(f"variant_{name}", [ds], standard_jobs(methods), SEEDS, 5, base_overrides=ov)

In [ ]:
for name in SELECTED:
    ds, ov, methods = VARIANTS[name]
    print(f"\n{name}  {ov}")
    display(table2_df(table2_numeric(f"variant_{name}", [ds], methods), [ds], methods))